# MongoDB & NoSQL — A Hands-On Course

This notebook is a self-contained, runnable introduction to MongoDB for people
who already know some Python. It covers:

1. NoSQL vs. relational databases — what's actually different
2. Installing MongoDB (summary — full steps are in the companion setup guide)
3. Connecting from Python with `pymongo`
4. Databases, collections, and documents
5. CRUD: Create, Read, Update, Delete
6. Query operators
7. Indexes
8. The aggregation framework (`$match`, `$group`, `$lookup`, ...)
9. Schema design: embedding vs. referencing
10. Exercises

**How to use this notebook:** run the cells top to bottom. Each section
builds on the last, so don't skip around on a first pass. If you have a real
MongoDB server running locally, the connection cell below will use it
automatically. If not, it transparently falls back to an in-memory mock
(`mongomock`) so every cell below still runs — the code is identical either
way, since `mongomock` implements the same `pymongo` API.


## 1. What is NoSQL, and what is MongoDB?

**Relational databases** (Postgres, MySQL, SQL Server) store data in rigid
tables: fixed columns, fixed types, and relationships expressed through
foreign keys and `JOIN`s. Every row in a table must have the same shape.

**NoSQL databases** relax those constraints. There are several families:

| Family | Example | Data model |
|---|---|---|
| Document | **MongoDB**, CouchDB | JSON-like documents grouped into collections |
| Key-value | Redis, DynamoDB | Simple key → value lookups |
| Wide-column | Cassandra, HBase | Rows with dynamic, sparse columns |
| Graph | Neo4j | Nodes and edges |

**MongoDB** is a *document database*. Instead of rows in a table, it stores
**documents** (think: a Python dict, serialized as BSON — binary JSON) inside
**collections** (think: a table, but schema-less). A whole "order" — its
items, shipping address, and customer notes — can live in one document
instead of being split across five joined tables.

### Why teams reach for it
- **Flexible schema** — documents in the same collection don't need identical
  fields. Good for evolving data models, semi-structured data, prototyping.
- **Natural mapping to objects** — a document often mirrors the object you'd
  build in code, which cuts down on ORM impedance mismatch.
- **Horizontal scaling** — built-in sharding for very large datasets.
- **Rich queries** — unlike plain key-value stores, MongoDB supports
  filtering, indexing, and a powerful aggregation pipeline.

### Trade-offs worth knowing
- No enforced foreign keys — referential integrity across collections is
  your responsibility (or use `$lookup`, covered below).
- Historically weaker multi-document transaction guarantees than relational
  databases (modern MongoDB *does* support multi-document ACID
  transactions, but it's less central to the model).
- Denormalized, embedded data can duplicate information — a deliberate
  trade-off of read speed for write complexity, not a bug.

**Rule of thumb:** if your data is naturally hierarchical/document-shaped and
your access patterns are known in advance, MongoDB tends to fit well. If you
need complex multi-table joins and strict schema enforcement, a relational
database is often the simpler choice.


## 2. Installing MongoDB (summary)

You have three practical options. Full step-by-step instructions for each
are in the companion file **`MongoDB_Setup_Guide.md`** — this is just the
short version so the notebook stands on its own.

**Option A — MongoDB Atlas (cloud, no install, recommended for a first pass)**
Create a free cluster at [mongodb.com/atlas](https://www.mongodb.com/atlas)
and copy its connection string (`mongodb+srv://...`). No local install
needed — good for classrooms where students can't install software.

**Option B — Local install (Community Edition)**
Install the `mongod` server natively on Windows, macOS, or Linux. Gives you
a real `localhost:27017` server. See the setup guide for OS-specific steps.

**Option C — Docker (fastest local option if Docker is already installed)**
```bash
docker run -d --name mongodb -p 27017:27017 mongo:8.0
```
One command, no system-level install, easy to tear down with
`docker rm -f mongodb`.

Whichever option you choose, once MongoDB is reachable at a connection
string, everything below works unchanged — only the URI in the next cell
needs to point at it.


## 3. Connecting from Python

MongoDB's official Python driver is `pymongo`. Install it with:

```bash
pip install pymongo
```

Then connect with a `MongoClient`. The cell below tries a real local server
first (`mongodb://localhost:27017/`) and falls back to an in-memory mock if
none is reachable, so this notebook works whether or not you've installed
MongoDB yet. When you do have a real server, just delete the `try`/`except`
fallback and use `MongoClient(...)` directly — the rest of the notebook
doesn't change.


In [1]:
from pymongo import MongoClient
from pymongo.errors import ServerSelectionTimeoutError

MONGO_URI = "mongodb://localhost:27017/"   # change this for Atlas / Docker / a remote host

try:
    client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=1500)
    client.admin.command("ping")
    print(f"Connected to a real MongoDB server at {MONGO_URI}")
except ServerSelectionTimeoutError:
    import mongomock
    client = mongomock.MongoClient()
    print("No local MongoDB server found — using an in-memory mock "
          "(mongomock) so the rest of this notebook still runs.\n"
          "Install MongoDB and re-run this cell to use a real server instead.")


No local MongoDB server found — using an in-memory mock (mongomock) so the rest of this notebook still runs.
Install MongoDB and re-run this cell to use a real server instead.


## 4. Databases, collections, documents

- A **client** connects to a MongoDB server (or cluster).
- A server hosts multiple **databases**.
- A database holds multiple **collections** (roughly: tables).
- A collection holds **documents** (roughly: rows — but each is a JSON-like
  object, and different documents in the same collection can have different
  fields).

Both databases and collections are created **lazily** — the first time you
write to them.


In [2]:
db = client["school"]              # database (created on first write)
students = db["students"]         # collection (created on first write)

print("Databases before insert:", client.list_database_names())


Databases before insert: []


## 5. Create — inserting documents

`insert_one` inserts a single document; `insert_many` inserts a list.
MongoDB auto-generates a unique `_id` (an `ObjectId`) if you don't supply
one — `_id` is the primary key of every document.

Notice the documents below don't all have the same fields. That's normal —
this is the flexible-schema property in action.


In [3]:
result = students.insert_one({
    "name": "Asha Verma",
    "age": 21,
    "major": "Computer Science",
    "gpa": 8.7,
    "courses": ["Databases", "Operating Systems", "Algorithms"],
})
print("Inserted _id:", result.inserted_id)


Inserted _id: 6a9323ca9e4ad7cc56472633


In [4]:
more_students = [
    {"name": "Rahul Singh", "age": 22, "major": "Computer Science",
     "gpa": 7.9, "courses": ["Databases", "Networks"]},
    {"name": "Emily Chen", "age": 20, "major": "Data Science",
     "gpa": 9.1, "courses": ["Statistics", "Machine Learning", "Databases"]},
    {"name": "Liam O'Connor", "age": 23, "major": "Data Science",
     "gpa": 8.2, "courses": ["Statistics", "Databases"],
     "scholarship": True},          # this field doesn't exist on other docs — that's fine
    {"name": "Priya Nair", "age": 21, "major": "Mathematics",
     "gpa": 8.9, "courses": ["Linear Algebra", "Statistics"]},
]
result = students.insert_many(more_students)
print(f"Inserted {len(result.inserted_ids)} documents")
print("Total documents in collection:", students.count_documents({}))


Inserted 4 documents
Total documents in collection: 5


## 6. Read — querying documents

`find_one` returns a single matching document (or `None`); `find` returns a
**cursor** you iterate over. An empty filter `{}` matches everything.


In [5]:
one = students.find_one({"name": "Asha Verma"})
print(one)


{'name': 'Asha Verma', 'age': 21, 'major': 'Computer Science', 'gpa': 8.7, 'courses': ['Databases', 'Operating Systems', 'Algorithms'], '_id': ObjectId('6a9323ca9e4ad7cc56472633')}


In [6]:
print("All students:")
for s in students.find():
    print(f"  {s['name']:<15} major={s['major']:<16} gpa={s['gpa']}")


All students:
  Asha Verma      major=Computer Science gpa=8.7
  Rahul Singh     major=Computer Science gpa=7.9
  Emily Chen      major=Data Science     gpa=9.1
  Liam O'Connor   major=Data Science     gpa=8.2
  Priya Nair      major=Mathematics      gpa=8.9


### Filtering with query operators

MongoDB filters use operators prefixed with `$`. Some of the most common:

| Operator | Meaning | Example |
|---|---|---|
| `$eq` | equals (default if you just give a value) | `{"major": "Data Science"}` |
| `$ne` | not equal | `{"major": {"$ne": "Data Science"}}` |
| `$gt` / `$gte` | greater than / or equal | `{"gpa": {"$gt": 8.5}}` |
| `$lt` / `$lte` | less than / or equal | `{"age": {"$lte": 21}}` |
| `$in` | value in a list | `{"major": {"$in": ["Data Science", "Mathematics"]}}` |
| `$nin` | value not in a list | `{"major": {"$nin": ["Physics"]}}` |
| `$exists` | field is present / absent | `{"scholarship": {"$exists": True}}` |
| `$regex` | pattern match | `{"name": {"$regex": "^A"}}` |
| `$and` / `$or` | combine conditions | `{"$or": [{"age": 20}, {"age": 23}]}` |


In [7]:
high_gpa = students.find({"gpa": {"$gt": 8.5}})
print("GPA > 8.5:")
for s in high_gpa:
    print(f"  {s['name']} — {s['gpa']}")


GPA > 8.5:
  Asha Verma — 8.7
  Emily Chen — 9.1
  Priya Nair — 8.9


In [8]:
ds_or_math = students.find({"major": {"$in": ["Data Science", "Mathematics"]}})
print("Data Science or Mathematics majors:")
for s in ds_or_math:
    print(f"  {s['name']} — {s['major']}")


Data Science or Mathematics majors:
  Emily Chen — Data Science
  Liam O'Connor — Data Science
  Priya Nair — Mathematics


In [9]:
scholars = students.find({"scholarship": {"$exists": True}})
print("Students with a 'scholarship' field at all:")
for s in scholars:
    print(f"  {s['name']}")


Students with a 'scholarship' field at all:
  Liam O'Connor


### Projection, sorting, limiting

The second argument to `find` is a **projection**: which fields to include
(`1`) or exclude (`0`). `_id` is included by default unless you exclude it
explicitly. `sort`, `limit`, and `skip` chain onto the cursor.


In [10]:
top_3 = (
    students.find({}, {"_id": 0, "name": 1, "gpa": 1})
            .sort("gpa", -1)   # -1 = descending, 1 = ascending
            .limit(3)
)
print("Top 3 by GPA:")
for s in top_3:
    print(f"  {s}")


Top 3 by GPA:
  {'name': 'Emily Chen', 'gpa': 9.1}
  {'name': 'Priya Nair', 'gpa': 8.9}
  {'name': 'Asha Verma', 'gpa': 8.7}


## 7. Update — modifying documents

Updates take a filter (which documents to touch) and an update document
built from operators like `$set`, `$inc`, `$push`, `$unset`. **Always use an
update operator** — `update_one({...}, {"gpa": 9.0})` without `$set` would
raise an error/replace the whole document depending on driver version, which
is almost never what you want.

- `update_one` — updates the first match.
- `update_many` — updates every match.
- `upsert=True` — insert a new document if nothing matches.


In [11]:
students.update_one(
    {"name": "Rahul Singh"},
    {"$set": {"gpa": 8.1}}
)
print(students.find_one({"name": "Rahul Singh"}, {"_id": 0}))


{'name': 'Rahul Singh', 'age': 22, 'major': 'Computer Science', 'gpa': 8.1, 'courses': ['Databases', 'Networks']}


In [12]:
students.update_many(
    {"major": "Computer Science"},
    {"$inc": {"age": 1}}      # everyone in CS has a birthday
)
for s in students.find({"major": "Computer Science"}, {"_id": 0, "name": 1, "age": 1}):
    print(s)


{'name': 'Asha Verma', 'age': 22}
{'name': 'Rahul Singh', 'age': 23}


In [13]:
students.update_one(
    {"name": "Asha Verma"},
    {"$push": {"courses": "Distributed Systems"}}   # append to an array field
)
print(students.find_one({"name": "Asha Verma"}, {"_id": 0, "name": 1, "courses": 1}))


{'name': 'Asha Verma', 'courses': ['Databases', 'Operating Systems', 'Algorithms', 'Distributed Systems']}


In [14]:
result = students.update_one(
    {"name": "Zara Ahmed"},                         # doesn't exist yet
    {"$set": {"age": 19, "major": "Physics", "gpa": 8.0}},
    upsert=True
)
print("Matched:", result.matched_count, "| Upserted _id:", result.upserted_id)


Matched: 0 | Upserted _id: 6a9323ca9e4ad7cc56472638


## 8. Delete — removing documents

`delete_one` removes the first match, `delete_many` removes every match.
Deletes are permanent — there's no built-in undo.


In [15]:
result = students.delete_one({"name": "Zara Ahmed"})
print("Deleted count:", result.deleted_count)


Deleted count: 1


In [16]:
result = students.delete_many({"gpa": {"$lt": 8.2}})   # drops Rahul Singh (gpa 8.1)
print("Deleted count:", result.deleted_count)
print("Remaining:", students.count_documents({}))


Deleted count: 1
Remaining: 4


## 9. Indexes

Without an index, MongoDB scans every document to satisfy a query (a
"collection scan"). An index lets it jump straight to matching documents —
the same idea as a book's index, and the same trade-off: faster reads, a
little slower writes, extra storage.

`_id` is indexed automatically. You create others with `create_index`.


In [17]:
students.create_index("gpa")                       # single-field index
students.create_index([("major", 1), ("gpa", -1)])   # compound index

print("Indexes on `students`:")
for name, info in students.index_information().items():
    print(f"  {name}: {info['key']}")


Indexes on `students`:
  _id_: [('_id', 1)]
  gpa_1: [('gpa', 1)]
  major_1_gpa_-1: [('major', 1), ('gpa', -1)]


In [18]:
# a *unique* index enforces no duplicates
students.create_index("name", unique=True)

try:
    students.insert_one({"name": "Priya Nair", "age": 30, "major": "Physics", "gpa": 5.0})
except Exception as e:
    print(f"Insert rejected — {type(e).__name__}: {e}")


Insert rejected — DuplicateKeyError: E11000 Duplicate Key Error


## 10. The aggregation framework

`find` answers "which documents match?". The **aggregation pipeline**
answers "how can I reshape, group, and summarize my data?" — it's
MongoDB's answer to `GROUP BY`, `JOIN`, and computed columns.

A pipeline is a list of **stages**, each transforming the documents that
flow out of the previous stage. Common stages:

| Stage | Purpose |
|---|---|
| `$match` | filter documents (like `find`) |
| `$group` | group by a key and compute aggregates (`$sum`, `$avg`, `$max`, `$min`, `$push`) |
| `$sort` | sort results |
| `$project` | reshape / compute new fields |
| `$limit` / `$skip` | pagination |
| `$unwind` | flatten an array field into one document per element |
| `$lookup` | a left-outer-join against another collection |


In [19]:
pipeline = [
    {"$group": {
        "_id": "$major",
        "avg_gpa": {"$avg": "$gpa"},
        "count": {"$sum": 1},
        "students": {"$push": "$name"},
    }},
    {"$sort": {"avg_gpa": -1}},
]

print("Average GPA by major:")
for doc in students.aggregate(pipeline):
    print(f"  {doc['_id']:<16} avg_gpa={doc['avg_gpa']:.2f}  n={doc['count']}  {doc['students']}")


Average GPA by major:
  Mathematics      avg_gpa=8.90  n=1  ['Priya Nair']
  Computer Science avg_gpa=8.70  n=1  ['Asha Verma']
  Data Science     avg_gpa=8.65  n=2  ['Emily Chen', "Liam O'Connor"]


### `$unwind` — one document per array element

`courses` is an array field. `$unwind` "explodes" it so each course gets
its own document, which makes it easy to count enrollment per course.


In [20]:
pipeline = [
    {"$unwind": "$courses"},
    {"$group": {"_id": "$courses", "enrolled": {"$sum": 1}}},
    {"$sort": {"enrolled": -1}},
]
print("Enrollment per course:")
for doc in students.aggregate(pipeline):
    print(f"  {doc['_id']:<22} {doc['enrolled']}")


Enrollment per course:
  Databases              3
  Statistics             3
  Algorithms             1
  Distributed Systems    1
  Linear Algebra         1
  Machine Learning       1
  Operating Systems      1


### `$lookup` — joining across collections

Let's add an `enrollments` collection referencing students by `_id`, and a
`courses` collection with extra metadata, then join them — this is the
document-database equivalent of a SQL `JOIN`.


In [21]:
asha = students.find_one({"name": "Asha Verma"})
emily = students.find_one({"name": "Emily Chen"})

courses_col = db["courses"]
courses_col.insert_many([
    {"_id": "Databases", "credits": 4, "department": "CS"},
    {"_id": "Machine Learning", "credits": 4, "department": "DS"},
    {"_id": "Statistics", "credits": 3, "department": "DS"},
])

enrollments = db["enrollments"]
enrollments.insert_many([
    {"student_id": asha["_id"], "course_id": "Databases", "grade": "A"},
    {"student_id": emily["_id"], "course_id": "Machine Learning", "grade": "A-"},
    {"student_id": emily["_id"], "course_id": "Statistics", "grade": "A"},
])

pipeline = [
    {"$lookup": {
        "from": "students",
        "localField": "student_id",
        "foreignField": "_id",
        "as": "student",
    }},
    {"$unwind": "$student"},
    {"$lookup": {
        "from": "courses",
        "localField": "course_id",
        "foreignField": "_id",
        "as": "course",
    }},
    {"$unwind": "$course"},
    {"$project": {
        "_id": 0,
        "student_name": "$student.name",
        "course": "$course_id",
        "credits": "$course.credits",
        "grade": "$grade",
    }},
]

print("Enrollment report (joined across 3 collections):")
for doc in enrollments.aggregate(pipeline):
    print(f"  {doc}")


Enrollment report (joined across 3 collections):
  {'student_name': 'Asha Verma', 'course': 'Databases', 'credits': 4, 'grade': 'A'}
  {'student_name': 'Emily Chen', 'course': 'Machine Learning', 'credits': 4, 'grade': 'A-'}
  {'student_name': 'Emily Chen', 'course': 'Statistics', 'credits': 3, 'grade': 'A'}


## 11. Schema design: embed or reference?

MongoDB gives you two ways to model a relationship, and choosing between
them is the main design decision in document modeling.

**Embedding** — put related data inside the parent document as a
sub-document or array.
```python
{
    "_id": 1,
    "title": "Order #1001",
    "items": [
        {"sku": "A1", "qty": 2, "price": 9.99},
        {"sku": "B7", "qty": 1, "price": 24.50},
    ],
}
```
Good when: the child data is only ever accessed with the parent, doesn't
grow unbounded, and doesn't need to be queried independently. One read
fetches everything — no join needed.

**Referencing** — store just an `_id`, and look the related document up
separately (optionally with `$lookup`), the same shape as the
`students` / `courses` / `enrollments` example above.
Good when: the related data is large, shared across many parents, updated
independently, or grows without bound (e.g. millions of enrollments
referencing a small `students` collection — embedding all of a popular
course's enrollments *inside* the course document would make it huge).

**Rule of thumb:** embed for "contains" / one-to-few relationships that are
always read together; reference for "relates to" / one-to-many-or-huge
relationships, or when child data needs to stand on its own.


## 12. Exercises

Try these yourself before checking the solutions cell.

1. Insert 3 new students of your choice, including at least one field none
   of the existing students have.
2. Find every student whose name starts with a vowel, using `$regex`.
3. Increase every Data Science student's GPA by `0.1` in one call.
4. Write an aggregation pipeline that returns the **oldest** student in each
   major.
5. Create a unique index on something other than `name` and prove it works
   by trying (and catching) a duplicate insert.


In [22]:
# --- your answers here ---


### Solutions (no peeking until you've tried!)

In [23]:
# 1.
students.insert_many([
    {"name": "Tom Baker", "age": 24, "major": "Physics", "gpa": 7.5, "year": "Senior"},
    {"name": "Nina Rao", "age": 20, "major": "Data Science", "gpa": 9.0, "year": "Sophomore"},
    {"name": "Sam Okafor", "age": 22, "major": "Mathematics", "gpa": 8.3, "year": "Junior"},
])

# 2.
vowel_names = students.find({"name": {"$regex": "^[AEIOU]", "$options": "i"}})
print([s["name"] for s in vowel_names])

# 3.
students.update_many({"major": "Data Science"}, {"$inc": {"gpa": 0.1}})

# 4.
pipeline = [
    {"$sort": {"age": -1}},
    {"$group": {"_id": "$major", "oldest_student": {"$first": "$name"}, "age": {"$first": "$age"}}},
]
print(list(students.aggregate(pipeline)))

# 5.
students.create_index("age")  # not unique by itself, so demonstrate on a synthetic unique field instead
db["courses"].create_index("_id", unique=True)  # _id is already implicitly unique — this just proves the point
try:
    db["courses"].insert_one({"_id": "Databases", "credits": 999, "department": "X"})
except Exception as e:
    print(f"Duplicate rejected — {type(e).__name__}: {e}")


['Asha Verma', 'Emily Chen']
[{'oldest_student': 'Asha Verma', 'age': 22, '_id': 'Computer Science'}, {'oldest_student': "Liam O'Connor", 'age': 23, '_id': 'Data Science'}, {'oldest_student': 'Sam Okafor', 'age': 22, '_id': 'Mathematics'}, {'oldest_student': 'Tom Baker', 'age': 24, '_id': 'Physics'}]
Duplicate rejected — DuplicateKeyError: E11000 Duplicate Key Error


## 13. Cheat sheet

```python
from pymongo import MongoClient
client = MongoClient("mongodb://localhost:27017/")
db = client["mydb"]
col = db["mycollection"]

# Create
col.insert_one({...})
col.insert_many([{...}, {...}])

# Read
col.find_one({"field": value})
col.find({"field": {"$gt": 10}}).sort("field", -1).limit(5)
col.count_documents({"field": value})
col.distinct("field")

# Update
col.update_one({"filter": ...}, {"$set": {"field": value}})
col.update_many({"filter": ...}, {"$inc": {"field": 1}}, upsert=True)

# Delete
col.delete_one({"filter": ...})
col.delete_many({"filter": ...})

# Indexes
col.create_index("field")
col.create_index([("field1", 1), ("field2", -1)], unique=True)

# Aggregation
col.aggregate([
    {"$match": {...}},
    {"$group": {"_id": "$field", "total": {"$sum": 1}}},
    {"$sort": {"total": -1}},
])
```

### Where to go next
- Official docs: <https://www.mongodb.com/docs/manual/>
- `pymongo` docs: <https://pymongo.readthedocs.io/>
- MongoDB University (free courses): <https://learn.mongodb.com/>
- MongoDB Compass — the official GUI, useful for browsing data visually
  alongside this notebook.
